In [68]:
!pip install scikit-learn datasets transformers Pillow rouge_score openai python-dotenv
!pip install openai
!pip install "accelerate>=1.1.0"
!pip istall transformers

ERROR: unknown command "istall" - maybe you meant "install"


# Projeto final - Paradigmas de Aprendizagem de máquina

## Montando base de dados

In [ ]:
from getting_database import buscar_filmes_terror

filmes1 = buscar_filmes_terror(quantidade=200, ordenar_por="vote_average.desc")
filmes2 = buscar_filmes_terror(quantidade=200, ordenar_por="popularity.desc")



### Salvar em arquivo

In [ ]:
import pandas as pd

df = pd.DataFrame(filmes1)
df2 = pd.DataFrame(filmes2)
df_concat = pd.concat([df, df2])
df_filtrado = df[["titulo", "poster", "sinopse"]]
df_filtrado.to_csv("filmes.csv", index=False, encoding="utf-8")
df = df_filtrado

## EDA e pré-processamento

In [4]:
import pandas as pd

df = pd.read_csv("filmes.csv")

In [5]:
df.head()

,titulo,poster,sinopse
0,O Jogo da Tentação,https://image.tmdb.org/t/p/w500/dtdZRfoNRNcH92...,"Um rapaz, que acabou de se tornar pai e luta c..."
1,Psicose,https://image.tmdb.org/t/p/w500/oC2iYT2on8c2iZ...,Marion Crane é uma secretária que rouba 40 mil...
2,La Leyenda de los Chaneques,https://image.tmdb.org/t/p/w500/4f9ghI3utknpeB...,NaN
3,Las leyendas: El origen,https://image.tmdb.org/t/p/w500/fR49hZdFJ6ZtRS...,NaN
4,Michael Jackson: Thriller,https://image.tmdb.org/t/p/w500/dYHGoPMkZMVuBA...,Uma noite no cinema se transforma em um pesade...


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   titulo   400 non-null    object
 1   poster   400 non-null    object
 2   sinopse  391 non-null    object
dtypes: object(3)
memory usage: 9.5+ KB


In [7]:
print("duplicados: ", df.duplicated().sum())
df = df.drop_duplicates()


duplicados:  66


In [8]:
print("Total de filmes: ", len(df))
df = df.dropna(subset=["sinopse"])
print("Total de filmes depois de remover sinopses nulas: ", len(df))


Total de filmes:  334
Total de filmes depois de remover sinopses nulas:  325


In [9]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Treino: {len(train_df)} filmes")
print(f"Teste:  {len(test_df)} filmes")

Treino: 260 filmes
Teste:  65 filmes


### Baixando pôsteres

Precisamos das imagens localmente para o modelo processá-las.
Cada pôster é baixado uma vez e salvo em `posters/`.

In [10]:
import requests
from pathlib import Path

POSTERS_DIR = Path("posters")
POSTERS_DIR.mkdir(exist_ok=True)

def download_poster(url):
    filename = url.split("/")[-1]
    path = POSTERS_DIR / filename
    if not path.exists():
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        path.write_bytes(r.content)
    return str(path)

def add_image_paths(df):
    paths = []
    for url in df["poster"]:
        try:
            paths.append(download_poster(url))
        except Exception:
            paths.append(None)
    df = df.copy()
    df["image_path"] = paths
    return df.dropna(subset=["image_path"]).reset_index(drop=True)

train_df = add_image_paths(train_df)
test_df = add_image_paths(test_df)
print(f"Treino: {len(train_df)}, Teste: {len(test_df)}")

Treino: 260, Teste: 65


### Convertendo para HuggingFace Dataset

O mesmo formato que o notebook BERT usa com `load_dataset("rotten_tomatoes")`.
Aqui construímos o dataset manualmente a partir do nosso DataFrame.

In [11]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["image_path", "titulo", "sinopse"]])
test_dataset = Dataset.from_pandas(test_df[["image_path", "titulo", "sinopse"]])
print(train_dataset)

/home/cecilia/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['image_path', 'titulo', 'sinopse'],
    num_rows: 260
})


## Carregando modelo

In [12]:
from transformers import BlipProcessor, BlipForConditionalGeneration

MODEL_CHECKPOINT = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(MODEL_CHECKPOINT)
model = BlipForConditionalGeneration.from_pretrained(MODEL_CHECKPOINT)

Loading weights: 100%|██████████| 473/473 [00:00<00:00, 30641.36it/s]
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Pré-processamento para o modelo

In [13]:
from PIL import Image as PILImage

def preprocess(example):
    image = PILImage.open(example["image_path"]).convert("RGB")
    inputs = processor(
        images=image,
        text="Gere uma sinopse de terror para o filme: " + example["titulo"],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=64,
    )
    labels = processor.tokenizer(
        example["sinopse"],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=128,
    ).input_ids
    labels[labels == processor.tokenizer.pad_token_id] = -100

    return {
        "pixel_values": inputs.pixel_values.squeeze(0),
        "input_ids": inputs.input_ids.squeeze(0),
        "attention_mask": inputs.attention_mask.squeeze(0),
        "labels": labels.squeeze(0),
    }

### Aplicando pré-processamento

Assim como no notebook BERT fazemos `.map(preprocessamento, batched=True)`, aqui aplicamos o mesmo padrão.
A diferença é que cada item carrega uma imagem além do texto.

In [14]:
tokenized_train = train_dataset.map(
    preprocess, batched=False,
    remove_columns=train_dataset.column_names,
)
tokenized_test = test_dataset.map(
    preprocess, batched=False,
    remove_columns=test_dataset.column_names,
)
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")
print(tokenized_train)

Map: 100%|██████████| 65/65 [00:00<00:00, 76.69 examples/s]

Dataset({
    features: ['pixel_values', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 260
})


## Definição de Métricas — LLM as a Judge

Como sinopses são textos criativos, métricas de sobreposição de palavras (como ROUGE) não capturam
bem a qualidade. Usamos um LLM para avaliar cada sinopse gerada em três critérios:
- **Clima de terror**: a sinopse transmite tensão/medo?
- **Coerência com o título**: faz sentido para o filme?
- **Qualidade narrativa**: é bem escrita e coesa?

As funções abaixo serão chamadas na seção de avaliação, após o treino.

In [15]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

JUDGE_PROMPT = """Você é um avaliador de sinopses de filmes de terror.

Título do filme: {titulo}
Sinopse gerada pelo modelo: {gerada}
Sinopse de referência: {referencia}

Avalie a sinopse gerada de 1 a 5 em cada critério e dê uma justificativa breve:
1. Clima de terror: a sinopse transmite tensão/medo?
2. Coerência com o título: faz sentido para o filme mencionado?
3. Qualidade narrativa: é bem escrita e coesa?

Responda APENAS neste formato JSON:
{{"clima_terror": <1-5>, "coerencia_titulo": <1-5>, "qualidade_narrativa": <1-5>, "justificativa": "<texto breve>"}}"""


def gerar_sinopse(row):
    inputs = processor(
        images=PILImage.open(row["image_path"]).convert("RGB"),
        text="Gere uma sinopse de terror para o filme: " + row["titulo"],
        return_tensors="pt",
    )
    output = model.generate(**inputs, max_length=128)
    return processor.decode(output[0], skip_special_tokens=True)


def avaliar_sinopse(titulo, sinopse_gerada, sinopse_referencia):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[{
            "role": "user",
            "content": JUDGE_PROMPT.format(
                titulo=titulo,
                gerada=sinopse_gerada,
                referencia=sinopse_referencia,
            ),
        }],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

In [16]:
import pandas as pd

def rodar_avaliacao(test_df):
    resultados = []
    for _, row in test_df.iterrows():
        sinopse_gerada = gerar_sinopse(row)
        avaliacao = avaliar_sinopse(row["titulo"], sinopse_gerada, row["sinopse"])
        resultados.append({
            "titulo": row["titulo"],
            "sinopse_gerada": sinopse_gerada,
            **avaliacao,
        })
    df_resultados = pd.DataFrame(resultados)
    print(df_resultados[["titulo", "clima_terror", "coerencia_titulo", "qualidade_narrativa", "justificativa"]].to_string())
    print("\n--- Médias ---")
    print(df_resultados[["clima_terror", "coerencia_titulo", "qualidade_narrativa"]].mean().round(2))
    return df_resultados

## Treinamento

Análogo à seção de `Treinamento` do notebook BERT, mas usamos `Seq2SeqTrainer`
em vez de `Trainer` porque a tarefa é geração de texto (seq2seq), não classificação.

Antes de treinar, verificamos se há GPU disponível e movemos o modelo para ela.

In [17]:
import torch
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Treinando em: {device}")

Treinando em: cpu


### Congelamento do Vision Encoder

Analogamente ao notebook BERT onde congelamos `bert.*` e treinamos só `classifier.*`,
aqui congelamos `vision_model.*` e treinamos só `text_decoder.*`.

O `vision_model` (ViT) já aprendeu a "ver" imagens em bilhões de exemplos — não precisamos
reaprender isso. O que queremos ensinar é o decoder a gerar texto no **estilo de terror**.

In [18]:
for name, param in model.named_parameters():
    if "vision_model" in name:
        param.requires_grad = False  # congela o ViT — ele já sabe "ver"

trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print(f"Parâmetros treináveis: {len(trainable)}")
print(f"Total de parâmetros:   {len(list(model.named_parameters()))}")

Parâmetros treináveis: 321
Total de parâmetros:   471


In [20]:
training_args = Seq2SeqTrainingArguments(
    output_dir="blip-horror",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    predict_with_generate=True,
    save_strategy="epoch",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=processor.tokenizer,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None, 'pad_token_id': 0}.
/home/cecilia/miniconda3/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


ValueError: Expected input batch_size (504) to match target batch_size (1016).